In [1]:
from dataclasses import dataclass
from typing import List, Optional, Tuple
import numpy as np
import pandas as pd


class BigMNumber:
    """Biểu diễn số học dạng: α*M + β phục vụ phương pháp Big-M."""

    def __init__(self, m: float = 0.0, c: float = 0.0) -> None:
        self.m = float(m)
        self.c = float(c)

    def __add__(self, other):
        if isinstance(other, (int, float)):
            return BigMNumber(self.m, self.c + other)
        return BigMNumber(self.m + other.m, self.c + other.c)

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        if isinstance(other, (int, float)):
            return BigMNumber(self.m, self.c - other)
        return BigMNumber(self.m - other.m, self.c - other.c)

    def __rsub__(self, other):
        if isinstance(other, (int, float)):
            return BigMNumber(-self.m, other - self.c)
        return BigMNumber(other.m - self.m, other.c - self.c)

    def __mul__(self, scalar: float):
        return BigMNumber(self.m * scalar, self.c * scalar)

    def __rmul__(self, scalar: float):
        return self.__mul__(scalar)

    def __gt__(self, other):
        o_m = other.m if isinstance(other, BigMNumber) else 0.0
        o_c = other.c if isinstance(other, BigMNumber) else float(other)
        if not np.isclose(self.m, o_m, atol=1e-8):
            return self.m > o_m
        return self.c > o_c + 1e-8

    def is_positive(self) -> bool:
        return self > 0.0

    def __str__(self) -> str:
        m_part = ""
        if not np.isclose(self.m, 0.0, atol=1e-8):
            m_round = int(round(self.m)) if np.isclose(self.m, round(self.m)) else self.m
            if m_round == 1:
                m_part = "M"
            elif m_round == -1:
                m_part = "-M"
            else:
                m_part = f"{m_round}M"

        c_round = int(round(self.c)) if np.isclose(self.c, round(self.c)) else round(self.c, 2)

        if not m_part:
            return str(c_round)
        if np.isclose(c_round, 0.0, atol=1e-8):
            return m_part
        return f"{m_part}+{c_round}" if c_round > 0 else f"{m_part}{c_round}"


@dataclass
class SimplexResult:
    status: str
    optimal_x: Optional[np.ndarray] = None
    optimal_value: Optional[float] = None
    iterations: int = 0


class PandasBigMSimplex:
    """Bộ giải Big-M Simplex hỗ trợ ẩn các cột được chỉ định ở các bảng tiếp theo."""

    def __init__(
        self,
        c: List[BigMNumber],
        A: np.ndarray,
        b: np.ndarray,
        basis_vars: List[int],
        var_names: List[str],
        precision: int = 5,
        drop_leaving_artificial: bool = True,  # Ẩn cột ẩn nhân tạo khi bị loại ra
    ) -> None:
        self.c = list(c)
        self.A = np.array(A, dtype=np.float64)
        self.b = np.array(b, dtype=np.float64)
        self.basis_vars = list(basis_vars)
        self.var_names = list(var_names)
        self.precision = precision
        self.num_constraints, self.num_vars = self.A.shape
        self.drop_leaving_artificial = drop_leaving_artificial
        # Tập hợp các chỉ số cột đang hiển thị
        self.active_cols = list(range(self.num_vars))

    def _calculate_deltas(self) -> Tuple[List[BigMNumber], BigMNumber]:
        """Tính f_0 và hệ số ước lượng Δ_j."""
        c_B = [self.c[idx] for idx in self.basis_vars]
        c_B_m = np.array([cb.m for cb in c_B])
        c_B_c = np.array([cb.c for cb in c_B])

        f_val = BigMNumber(
            m=float(np.dot(c_B_m, self.b)),
            c=float(np.dot(c_B_c, self.b))
        )

        term_m = np.dot(c_B_m, self.A)
        term_c = np.dot(c_B_c, self.A)
        deltas = [
            BigMNumber(m=term_m[j] - self.c[j].m, c=term_c[j] - self.c[j].c)
            for j in range(self.num_vars)
        ]
        return deltas, f_val

    def _build_dataframe(
        self,
        deltas: List[BigMNumber],
        f_val: BigMNumber,
        lambdas: Optional[np.ndarray] = None,
    ) -> pd.DataFrame:
        """Tạo DataFrame chỉ chứa các cột đang trong active_cols."""
        rows = []
        p = self.precision
        
        for i in range(self.num_constraints):
            row = {
                "Hệ số": self.c[self.basis_vars[i]].to_formatted_str(p),
                "Cơ sở": self.var_names[self.basis_vars[i]],
                "Phương án": f"{self.b[i]:.{p}f}",
            }
            for j in self.active_cols:
                row[self.var_names[j]] = f"{self.A[i, j]:.{p}f}"
            if lambdas is not None:
                row["λ"] = f"{lambdas[i]:.{p}f}" if not np.isnan(lambdas[i]) else "-"
            rows.append(row)

        delta_row = {
            "Hệ số": "",
            "Cơ sở": "Δ_j",
            "Phương án": str(f_val),
        }
        for j in self.active_cols:
            delta_row[self.var_names[j]] = str(deltas[j])
        if lambdas is not None:
            delta_row["λ"] = ""
        rows.append(delta_row)

        return pd.DataFrame(rows)

    def solve(self, max_iters: int = 15, verbose: bool = True) -> SimplexResult:
        pd.set_option("display.width", 1000)
        pd.set_option("display.max_columns", None)

        for step in range(max_iters):
            deltas, f_val = self._calculate_deltas()

            # Tìm max(Δ_j > 0) trong số các cột đang active
            positive_deltas = [j for j in self.active_cols if deltas[j].is_positive()]

            if not positive_deltas:
                if verbose:
                    print(f"\n{'='*30} BẢNG ĐƠN HÌNH {step} (TỐI ƯU) {'='*30}")
                    print(self._build_dataframe(deltas, f_val).to_string(index=False))
                    print("\n[+] ĐÃ ĐẠT ĐIỀU KIỆN TỐI ƯU (Tất cả Δ_j <= 0).")

                optimal_x = np.zeros(self.num_vars)
                for i, idx in enumerate(self.basis_vars):
                    optimal_x[idx] = self.b[i]

                return SimplexResult(
                    status="optimal",
                    optimal_x=optimal_x,
                    optimal_value=f_val.c,
                    iterations=step,
                )

            # Chọn cột chủ yếu p
            p = positive_deltas[0]
            for j in positive_deltas[1:]:
                if deltas[j] > deltas[p]:
                    p = j

            # Tính thương số λ_i = b_i / a_ip với a_ip > 0
            lambdas = np.full(self.num_constraints, np.nan)
            valid_rows = np.where(self.A[:, p] > 1e-8)[0]

            if len(valid_rows) == 0:
                if verbose:
                    print(f"\n[-] Cột {self.var_names[p]} không có hệ số a_ip > 0. Bài toán không bị chặn!")
                return SimplexResult(status="unbounded", iterations=step)

            lambdas[valid_rows] = self.b[valid_rows] / self.A[valid_rows, p]

            if verbose:
                print(f"\n{'='*35} BẢNG ĐƠN HÌNH {step} {'='*35}")
                print(self._build_dataframe(deltas, f_val, lambdas).to_string(index=False))

            # Chọn dòng chủ yếu q
            q = int(valid_rows[np.argmin(lambdas[valid_rows])])
            leaving_var_idx = self.basis_vars[q]
            pivot_val = self.A[q, p]

            if verbose:
                print(f"[>] Ẩn đưa vào: {self.var_names[p]} (Δ = {deltas[p]})")
                print(f"[<] Ẩn đưa ra: {self.var_names[leaving_var_idx]} (min λ = {lambdas[q]:.2g})")
                print(f"[*] Phần tử trục xoay: {pivot_val:.2g}")

            # Phép biến đổi khử Gauss-Jordan
            self.A[q] /= pivot_val
            self.b[q] /= pivot_val

            factors = self.A[:, p].copy()
            factors[q] = 0.0

            self.A -= factors[:, np.newaxis] * self.A[q]
            self.b -= factors * self.b[q]

            # Đổi ẩn cơ sở
            self.basis_vars[q] = p

            # LOẠI BỎ CỘT KHỎI BẢNG TIẾP THEO:
            # 1. Nếu ẩn đưa ra là ẩn nhân tạo (có M), không hiển thị ở bảng sau:
            if self.drop_leaving_artificial and self.c[leaving_var_idx].m > 0:
                if leaving_var_idx in self.active_cols:
                    self.active_cols.remove(leaving_var_idx)
                    if verbose:
                        print(f"[-] Ẩn nhân tạo {self.var_names[leaving_var_idx]} đã rời cơ sở -> Không hiển thị ở bảng sau.")

            # 2. Hoặc nếu bạn muốn ẩn ngay cột chủ yếu đưa vào:
            # self.active_cols.remove(p)

        raise TimeoutError("Đạt giới hạn vòng lặp tối đa.")

In [2]:
def main():
    var_names = ["x1", "x2", "x3", "x4", "x5", "x6", "x7", "x8"]

    # Hệ số hàm mục tiêu min f(x)
    c = [
        BigMNumber(m=0, c=4),
        BigMNumber(m=0, c=3),
        BigMNumber(m=0, c=0),
        BigMNumber(m=0, c=0),
        BigMNumber(m=0, c=0),   # x5 (ẩn phụ)
        BigMNumber(m=0, c=0),
        BigMNumber(m=1, c=0),   # x7 (ẩn nhân tạo M)
        BigMNumber(m=1, c=0),   # x8 (ẩn nhân tạo M)
    ]

    # Ma trận A tại Bảng 0
    A = np.array([
        [ 2.0, 1.0, -1.0, 0.0, 0.0, 0.0, 1.0, 0],
        [ 2.0, 5.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0],
        [ 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
        [ 0.0, 1.0, 0.0, 0, 0.0, 1.0, 0.0, 0.0],
    ])

    # Cột phương án b
    b = np.array([14.0, 30.0, 10.0, 9.0])

    # Cơ sở xuất phát: x6, x5, x7
    basis_vars = [6, 7, 4, 5]

    solver = PandasBigMSimplex(
        c, A, b, basis_vars, var_names, drop_leaving_artificial=True
    )
    res = solver.solve()

    if res.status == "optimal":
        print("\n" + "=" * 25 + " NGHIỆM TỐI ƯU " + "=" * 25)
        for name, val in zip(var_names[:5], res.optimal_x[:5]):
            print(f"  {name} = {val:.5f}")
        print(f"  Giá trị tối ưu f_min = {res.optimal_value:.5f}")


if __name__ == "__main__":
    main()


=================================== BẢNG ĐƠN HÌNH 0 ===================================


AttributeError: 'BigMNumber' object has no attribute 'to_formatted_str'

In [4]:
def main():
    var_names = ["x1", "x2", "x3", "x4", "x5", "x6"]

    # Hệ số hàm mục tiêu min f(x)
    c = [
        BigMNumber(m=0, c=11),
        BigMNumber(m=0, c=5),
        BigMNumber(m=0, c=0),
        BigMNumber(m=1, c=0),
        BigMNumber(m=0, c=0),   # x5 (ẩn phụ)
        BigMNumber(m=0, c=0),

    ]

    # Ma trận A tại Bảng 0
    A = np.array([
        [ 2.0, 1.0, -1.0, 1.0, 0.0, 0.0],
        [ 3.0, 4.0, 0.0, 0.0, 1.0, 0.0],
        [ 5.0, 4.0, 0.0, 0.0, 0.0, 1.0],

    ])

    # Cột phương án b
    b = np.array([20.0, 60.0, 80.0])

    # Cơ sở xuất phát: x6, x5, x7
    basis_vars = [3, 4, 5]

    solver = PandasBigMSimplex(
        c, A, b, basis_vars, var_names, drop_leaving_artificial=True
    )
    res = solver.solve()

    if res.status == "optimal":
        print("\n" + "=" * 25 + " NGHIỆM TỐI ƯU " + "=" * 25)
        for name, val in zip(var_names[:5], res.optimal_x[:5]):
            print(f"  {name} = {val:.5f}")
        print(f"  Giá trị tối ưu f_min = {res.optimal_value:.5f}")


if __name__ == "__main__":
    main()


=================================== BẢNG ĐƠN HÌNH 0 ===================================


AttributeError: 'BigMNumber' object has no attribute 'to_formatted_str'

In [3]:
from dataclasses import dataclass
from typing import List, Optional, Tuple
import numpy as np
import pandas as pd


class BigMNumber:
    """Biểu diễn số học dạng: α*M + β phục vụ phương pháp Big-M."""

    def __init__(self, m: float = 0.0, c: float = 0.0) -> None:
        self.m = float(m)
        self.c = float(c)

    def __add__(self, other):
        if isinstance(other, (int, float)):
            return BigMNumber(self.m, self.c + other)
        return BigMNumber(self.m + other.m, self.c + other.c)

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        if isinstance(other, (int, float)):
            return BigMNumber(self.m, self.c - other)
        return BigMNumber(self.m - other.m, self.c - other.c)

    def __rsub__(self, other):
        if isinstance(other, (int, float)):
            return BigMNumber(-self.m, other - self.c)
        return BigMNumber(other.m - self.m, other.c - self.c)

    def __mul__(self, scalar: float):
        return BigMNumber(self.m * scalar, self.c * scalar)

    def __rmul__(self, scalar: float):
        return self.__mul__(scalar)

    def __gt__(self, other):
        o_m = other.m if isinstance(other, BigMNumber) else 0.0
        o_c = other.c if isinstance(other, BigMNumber) else float(other)
        if not np.isclose(self.m, o_m, atol=1e-8):
            return self.m > o_m
        return self.c > o_c + 1e-8

    def is_positive(self) -> bool:
        return self > 0.0

    def to_formatted_str(self, decimals: int = 5) -> str:
        """Định dạng số theo số chữ số thập phân yêu cầu."""
        m_part = ""
        if not np.isclose(self.m, 0.0, atol=1e-8):
            m_round = round(self.m, decimals)
            if np.isclose(m_round, 1.0):
                m_part = "M"
            elif np.isclose(m_round, -1.0):
                m_part = "-M"
            elif m_round.is_integer():
                m_part = f"{int(m_round)}M"
            else:
                m_part = f"{m_round:.{decimals}f}M"

        c_round = round(self.c, decimals)
        c_str = f"{c_round:.{decimals}f}"

        if not m_part:
            return c_str
        if np.isclose(c_round, 0.0, atol=1e-8):
            return m_part
        if c_round > 0:
            return f"{m_part}+{c_str}"
        return f"{m_part}{c_str}"

    def __str__(self) -> str:
        return self.to_formatted_str(decimals=5)


@dataclass
class SimplexResult:
    status: str
    optimal_x: Optional[np.ndarray] = None
    optimal_value: Optional[float] = None
    iterations: int = 0


class PandasBigMSimplex:
    """Bộ giải Big-M Simplex hỗ trợ định dạng số thập phân thứ 5."""

    def __init__(
        self,
        c: List[BigMNumber],
        A: np.ndarray,
        b: np.ndarray,
        basis_vars: List[int],
        var_names: List[str],
        drop_leaving_artificial: bool = True,
        precision: int = 5,
    ) -> None:
        self.c = list(c)
        self.A = np.array(A, dtype=np.float64)
        self.b = np.array(b, dtype=np.float64)
        self.basis_vars = list(basis_vars)
        self.var_names = list(var_names)
        self.num_constraints, self.num_vars = self.A.shape
        self.drop_leaving_artificial = drop_leaving_artificial
        self.precision = precision
        self.active_cols = list(range(self.num_vars))

    def _calculate_deltas(self) -> Tuple[List[BigMNumber], BigMNumber]:
        c_B = [self.c[idx] for idx in self.basis_vars]
        c_B_m = np.array([cb.m for cb in c_B])
        c_B_c = np.array([cb.c for cb in c_B])

        f_val = BigMNumber(
            m=float(np.dot(c_B_m, self.b)),
            c=float(np.dot(c_B_c, self.b))
        )

        term_m = np.dot(c_B_m, self.A)
        term_c = np.dot(c_B_c, self.A)
        deltas = [
            BigMNumber(m=term_m[j] - self.c[j].m, c=term_c[j] - self.c[j].c)
            for j in range(self.num_vars)
        ]
        return deltas, f_val

    def _build_dataframe(
        self,
        deltas: List[BigMNumber],
        f_val: BigMNumber,
        lambdas: Optional[np.ndarray] = None,
    ) -> pd.DataFrame:
        p = self.precision
        rows = []
        for i in range(self.num_constraints):
            row = {
                "Hệ số": self.c[self.basis_vars[i]].to_formatted_str(p),
                "Cơ sở": self.var_names[self.basis_vars[i]],
                "Phương án": f"{self.b[i]:.{p}f}",
            }
            for j in self.active_cols:
                row[self.var_names[j]] = f"{self.A[i, j]:.{p}f}"
            if lambdas is not None:
                row["λ"] = f"{lambdas[i]:.{p}f}" if not np.isnan(lambdas[i]) else "-"
            rows.append(row)

        delta_row = {
            "Hệ số": "",
            "Cơ sở": "Δ_j",
            "Phương án": f_val.to_formatted_str(p),
        }
        for j in self.active_cols:
            delta_row[self.var_names[j]] = deltas[j].to_formatted_str(p)
        if lambdas is not None:
            delta_row["λ"] = ""
        rows.append(delta_row)

        return pd.DataFrame(rows)

    def solve(self, max_iters: int = 15, verbose: bool = True) -> SimplexResult:
        pd.set_option("display.width", 1000)
        pd.set_option("display.max_columns", None)

        for step in range(max_iters):
            deltas, f_val = self._calculate_deltas()
            positive_deltas = [j for j in self.active_cols if deltas[j].is_positive()]

            if not positive_deltas:
                if verbose:
                    print(f"\n{'='*35} BẢNG ĐƠN HÌNH {step} (TỐI ƯU) {'='*35}")
                    print(self._build_dataframe(deltas, f_val).to_string(index=False))
                    print("\n[+] ĐÃ ĐẠT ĐIỀU KIỆN TỐI ƯU (Tất cả Δ_j <= 0).")

                optimal_x = np.zeros(self.num_vars)
                for i, idx in enumerate(self.basis_vars):
                    optimal_x[idx] = self.b[i]

                return SimplexResult(
                    status="optimal",
                    optimal_x=optimal_x,
                    optimal_value=f_val.c,
                    iterations=step,
                )

            # Chọn cột chủ yếu p
            p = positive_deltas[0]
            for j in positive_deltas[1:]:
                if deltas[j] > deltas[p]:
                    p = j

            # Tính λ_i
            lambdas = np.full(self.num_constraints, np.nan)
            valid_rows = np.where(self.A[:, p] > 1e-8)[0]

            if len(valid_rows) == 0:
                if verbose:
                    print(f"\n[-] Cột {self.var_names[p]} không có hệ số a_ip > 0. Bài toán không bị chặn!")
                return SimplexResult(status="unbounded", iterations=step)

            lambdas[valid_rows] = self.b[valid_rows] / self.A[valid_rows, p]

            if verbose:
                print(f"\n{'='*40} BẢNG ĐƠN HÌNH {step} {'='*40}")
                print(self._build_dataframe(deltas, f_val, lambdas).to_string(index=False))

            # Chọn dòng chủ yếu q
            q = int(valid_rows[np.argmin(lambdas[valid_rows])])
            leaving_var_idx = self.basis_vars[q]
            pivot_val = self.A[q, p]

            if verbose:
                print(f"[>] Ẩn đưa vào: {self.var_names[p]} (Δ = {deltas[p].to_formatted_str(self.precision)})")
                print(f"[<] Ẩn đưa ra: {self.var_names[leaving_var_idx]} (min λ = {lambdas[q]:.{self.precision}f})")
                print(f"[*] Phần tử trục: {pivot_val:.{self.precision}f}")

            # Phép khử Gauss-Jordan
            self.A[q] /= pivot_val
            self.b[q] /= pivot_val

            factors = self.A[:, p].copy()
            factors[q] = 0.0

            self.A -= factors[:, np.newaxis] * self.A[q]
            self.b -= factors * self.b[q]

            self.basis_vars[q] = p

            if self.drop_leaving_artificial and self.c[leaving_var_idx].m > 0:
                if leaving_var_idx in self.active_cols:
                    self.active_cols.remove(leaving_var_idx)
                    if verbose:
                        print(f"[-] Ẩn nhân tạo {self.var_names[leaving_var_idx]} đã rời cơ sở -> Không hiển thị ở bảng sau.")

        raise TimeoutError("Đạt giới hạn vòng lặp tối đa.")


In [4]:
def main():
    var_names = ["x1", "x2", "x3", "x4", "x5", "x6", "x7", "x8"]

    # Hệ số hàm mục tiêu min f(x)
    c = [
        BigMNumber(m=0, c=4),
        BigMNumber(m=0, c=3),
        BigMNumber(m=0, c=0),
        BigMNumber(m=0, c=0),
        BigMNumber(m=0, c=0),   # x5 (ẩn phụ)
        BigMNumber(m=0, c=0),
        BigMNumber(m=1, c=0),   # x7 (ẩn nhân tạo M)
        BigMNumber(m=1, c=0),   # x8 (ẩn nhân tạo M)
    ]

    # Ma trận A tại Bảng 0
    A = np.array([
        [ 2.0, 1.0, -1.0, 0.0, 0.0, 0.0, 1.0, 0],
        [ 2.0, 5.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0],
        [ 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
        [ 0.0, 1.0, 0.0, 0, 0.0, 1.0, 0.0, 0.0],
    ])

    # Cột phương án b
    b = np.array([14.0, 30.0, 10.0, 9.0])

    # Cơ sở xuất phát: x6, x5, x7
    basis_vars = [6, 7, 4, 5]

    solver = PandasBigMSimplex(
        c, A, b, basis_vars, var_names, drop_leaving_artificial=True
    )
    res = solver.solve()

    if res.status == "optimal":
        print("\n" + "=" * 25 + " NGHIỆM TỐI ƯU " + "=" * 25)
        for name, val in zip(var_names[:5], res.optimal_x[:5]):
            print(f"  {name} = {val:.5f}")
        print(f"  Giá trị tối ưu f_min = {res.optimal_value:.5f}")


if __name__ == "__main__":
    main()


======================================== BẢNG ĐƠN HÌNH 0 ========================================
  Hệ số Cơ sở Phương án         x1         x2       x3       x4      x5      x6      x7      x8        λ
      M    x7  14.00000    2.00000    1.00000 -1.00000  0.00000 0.00000 0.00000 1.00000 0.00000 14.00000
      M    x8  30.00000    2.00000    5.00000  0.00000 -1.00000 0.00000 0.00000 0.00000 1.00000  6.00000
0.00000    x5  10.00000    1.00000    0.00000  0.00000  0.00000 1.00000 0.00000 0.00000 0.00000        -
0.00000    x6   9.00000    0.00000    1.00000  0.00000  0.00000 0.00000 1.00000 0.00000 0.00000  9.00000
          Δ_j       44M 4M-4.00000 6M-3.00000       -M       -M 0.00000 0.00000 0.00000 0.00000         
[>] Ẩn đưa vào: x2 (Δ = 6M-3.00000)
[<] Ẩn đưa ra: x8 (min λ = 6.00000)
[*] Phần tử trục: 5.00000
[-] Ẩn nhân tạo x8 đã rời cơ sở -> Không hiển thị ở bảng sau.

======================================== BẢNG ĐƠN HÌNH 1 ========================================
  Hệ số Cơ s

In [6]:
def main():
    var_names = ["x1", "x2", "x3", "x41", "x42", "x5"]

    # Hệ số hàm mục tiêu min f(x)
    c = [
        BigMNumber(m=0, c=1),
        BigMNumber(m=0, c=-1),
        BigMNumber(m=0, c=3),
        BigMNumber(m=0, c=2),
        BigMNumber(m=0, c=-2),   # x5 (ẩn phụ)
        BigMNumber(m=0, c=0),
        # BigMNumber(m=1, c=0),   # x7 (ẩn nhân tạo M)
        # BigMNumber(m=1, c=0),   # x8 (ẩn nhân tạo M)
    ]

    # Ma trận A tại Bảng 0
    A = np.array([
        [ 1.0, 2.0, -1.0, 0.0, 0.0, 0.0],
        [ 0.0, 1.0, 1.0, 1.0, -1.0, 0.0],
        [ 0.0, 1.0, 1.0, 0.0, 0.0, 1.0],
    ])

    # Cột phương án b
    b = np.array([18.0, 12.0, 15.0])

    # Cơ sở xuất phát: x6, x5, x7
    basis_vars = [0, 3, 5]

    solver = PandasBigMSimplex(
        c, A, b, basis_vars, var_names, drop_leaving_artificial=True
    )
    res = solver.solve()

    if res.status == "optimal":
        print("\n" + "=" * 25 + " NGHIỆM TỐI ƯU " + "=" * 25)
        for name, val in zip(var_names[:5], res.optimal_x[:5]):
            print(f"  {name} = {val:.5f}")
        print(f"  Giá trị tối ưu f_min = {res.optimal_value:.5f}")


if __name__ == "__main__":
    main()


======================================== BẢNG ĐƠN HÌNH 0 ========================================
  Hệ số Cơ sở Phương án      x1      x2       x3     x41      x42      x5        λ
1.00000    x1  18.00000 1.00000 2.00000 -1.00000 0.00000  0.00000 0.00000  9.00000
2.00000   x41  12.00000 0.00000 1.00000  1.00000 1.00000 -1.00000 0.00000 12.00000
0.00000    x5  15.00000 0.00000 1.00000  1.00000 0.00000  0.00000 1.00000 15.00000
          Δ_j  42.00000 0.00000 5.00000 -2.00000 0.00000  0.00000 0.00000         
[>] Ẩn đưa vào: x2 (Δ = 5.00000)
[<] Ẩn đưa ra: x1 (min λ = 9.00000)
[*] Phần tử trục: 2.00000

======================================== BẢNG ĐƠN HÌNH 1 ========================================
   Hệ số Cơ sở Phương án       x1      x2       x3     x41      x42      x5       λ
-1.00000    x2   9.00000  0.50000 1.00000 -0.50000 0.00000  0.00000 0.00000       -
 2.00000   x41   3.00000 -0.50000 0.00000  1.50000 1.00000 -1.00000 0.00000 2.00000
 0.00000    x5   6.00000 -0.50000 0.0000

In [ ]:
import numpy as np


def main():
    # 9 biến theo đúng thứ tự bài toán: y1, y21, y22, y'3, y4, y5, y6, y7, y8
    var_names = ["y1", "y2", "y3", "y4", "y5", "y6", "y7", "y8", "y9"]

    # Hệ số hàm mục tiêu min: 10y1 + 12y21 - 12y22 + 11y'3 + 0y4 + 0y5 + My6 + My7 + My8
    c = [
        BigMNumber(m=0, c=3),      # y1
        BigMNumber(m=0, c=2),      # y21
        BigMNumber(m=0, c=4),     # y22
        BigMNumber(m=0, c=0),      # y'3
        BigMNumber(m=0, c=0),       # y4
        BigMNumber(m=0, c=0),       # y5
        BigMNumber(m=1, c=0),       # y6 (ẩn nhân tạo M)
        BigMNumber(m=1, c=0),       # y7 (ẩn nhân tạo M)
        BigMNumber(m=1, c=0),       # y8 (ẩn nhân tạo M)
    ]

    # Ma trận hệ số ràng buộc A (kích thước 3 x 9)
    A = np.array([
        [2.0, -2.0,  2.0, -1.0, -1.0,  0.0, 1.0, 0.0, 0.0],  # Dòng 1: = 1
        [3.0, -1.0,  1.0,  3.0,  0.0, -1.0, 0.0, 1.0, 0.0],  # Dòng 2: = 2
        [1.0,  3.0, -3.0,  1.0,  0.0,  0.0, 0.0, 0.0, 1.0],  # Dòng 3: = 3
    ])

    # Véc-tơ vế phải b
    b = np.array([1.0, 2.0, 3.0])

    # Cơ sở xuất phát đơn vị ban đầu là các ẩn nhân tạo [y6, y7, y8] (chỉ số tương ứng 6, 7, 8)
    basis_vars = [6, 7, 8]

    solver = PandasBigMSimplex(
        c, A, b, basis_vars, var_names, drop_leaving_artificial=True
    )
    res = solver.solve()

    if res.status == "optimal":
        print("\n" + "=" * 25 + " NGHIỆM TỐI ƯU " + "=" * 25)
        for name, val in zip(var_names, res.optimal_x):
            print(f"  {name} = {val:.5f}")
        
        # Khôi phục giá trị các biến gốc ban đầu y2 và y3:
        # y2 = y21 - y22
        # y3 = -y'3 (vì y'3 = -y3 >= 0)
        y2 = res.optimal_x[1] - res.optimal_x[2]
        y3 = -res.optimal_x[3]
        print("-" * 55)
        print(f"  y1 = {res.optimal_x[0]:.5f}")
        print(f"  y2 = {y2:.5f}")
        print(f"  y3 = {y3:.5f}")
        print(f"  Giá trị tối ưu f_min = {res.optimal_value:.5f}")


if __name__ == "__main__":
    main()


======================================== BẢNG ĐƠN HÌNH 0 ========================================
Hệ số Cơ sở Phương án          y1       y21      y22    y3_prime       y4       y5      y6      y7      y8       λ
    M    y6   1.00000     2.00000  -2.00000  2.00000    -1.00000 -1.00000  0.00000 1.00000 0.00000 0.00000 0.50000
    M    y7   2.00000     3.00000  -1.00000  1.00000     3.00000  0.00000 -1.00000 0.00000 1.00000 0.00000 0.66667
    M    y8   3.00000     1.00000   3.00000 -3.00000     1.00000  0.00000  0.00000 0.00000 0.00000 1.00000 3.00000
        Δ_j        6M 6M-10.00000 -12.00000 12.00000 3M-11.00000       -M       -M 0.00000 0.00000 0.00000        
[>] Ẩn đưa vào: y1 (Δ = 6M-10.00000)
[<] Ẩn đưa ra: y6 (min λ = 0.50000)
[*] Phần tử trục: 2.00000
[-] Ẩn nhân tạo y6 đã rời cơ sở -> Không hiển thị ở bảng sau.

======================================== BẢNG ĐƠN HÌNH 1 ========================================
   Hệ số Cơ sở  Phương án      y1         y21          y22    y3_p

In [6]:
import numpy as np


def main():
    # 9 biến theo đúng thứ tự bài toán: y1, y21, y22, y'3, y4, y5, y6, y7, y8
    var_names = ["y1", "y2", "y3", "y4", "y5", "y6", "y7", "y8", "y9"]

    # Hệ số hàm mục tiêu min: 10y1 + 12y21 - 12y22 + 11y'3 + 0y4 + 0y5 + My6 + My7 + My8
    c = [
        BigMNumber(m=0, c=3),      # y1
        BigMNumber(m=0, c=2),      # y21
        BigMNumber(m=0, c=4),     # y22
        BigMNumber(m=0, c=0),      # y'3
        BigMNumber(m=0, c=0),       # y4
        BigMNumber(m=0, c=0),       # y5
        BigMNumber(m=1, c=0),       # y6 (ẩn nhân tạo M)
        BigMNumber(m=1, c=0),       # y7 (ẩn nhân tạo M)
        BigMNumber(m=1, c=0),       # y8 (ẩn nhân tạo M)
    ]

    # Ma trận hệ số ràng buộc A (kích thước 3 x 9)
    A = np.array([
        [3.0, 4.0,  1.0, -1.0, 0.0,  0.0, 1.0, 0.0, 0.0],  # Dòng 1: = 1
        [4.0, 1.0,  3.0,  0.0, -1.0, 0.0, 0.0, 1.0, 0.0],  # Dòng 2: = 2
        [2.0,  2.0, 2.0,  0.0,  0.0, -1.0, 0.0, 0.0, 1.0],  # Dòng 3: = 3
    ])

    # Véc-tơ vế phải b
    b = np.array([2, 4, 3])

    # Cơ sở xuất phát đơn vị ban đầu là các ẩn nhân tạo [y6, y7, y8] (chỉ số tương ứng 6, 7, 8)
    basis_vars = [6, 7, 8]

    solver = PandasBigMSimplex(
        c, A, b, basis_vars, var_names, drop_leaving_artificial=True
    )
    res = solver.solve()

    if res.status == "optimal":
        print("\n" + "=" * 25 + " NGHIỆM TỐI ƯU " + "=" * 25)
        for name, val in zip(var_names, res.optimal_x):
            print(f"  {name} = {val:.5f}")
        
        # Khôi phục giá trị các biến gốc ban đầu y2 và y3:
        # y2 = y21 - y22
        # y3 = -y'3 (vì y'3 = -y3 >= 0)
        y2 = res.optimal_x[1] - res.optimal_x[2]
        y3 = -res.optimal_x[3]
        print("-" * 55)
        print(f"  y1 = {res.optimal_x[0]:.5f}")
        print(f"  y2 = {y2:.5f}")
        print(f"  y3 = {y3:.5f}")
        print(f"  Giá trị tối ưu f_min = {res.optimal_value:.5f}")


if __name__ == "__main__":
    main()


======================================== BẢNG ĐƠN HÌNH 0 ========================================
Hệ số Cơ sở Phương án         y1         y2         y3       y4       y5       y6      y7      y8      y9       λ
    M    y7   2.00000    3.00000    4.00000    1.00000 -1.00000  0.00000  0.00000 1.00000 0.00000 0.00000 0.66667
    M    y8   4.00000    4.00000    1.00000    3.00000  0.00000 -1.00000  0.00000 0.00000 1.00000 0.00000 1.00000
    M    y9   3.00000    2.00000    2.00000    2.00000  0.00000  0.00000 -1.00000 0.00000 0.00000 1.00000 1.50000
        Δ_j        9M 9M-3.00000 7M-2.00000 6M-4.00000       -M       -M       -M 0.00000 0.00000 0.00000        
[>] Ẩn đưa vào: y1 (Δ = 9M-3.00000)
[<] Ẩn đưa ra: y7 (min λ = 0.66667)
[*] Phần tử trục: 3.00000
[-] Ẩn nhân tạo y7 đã rời cơ sở -> Không hiển thị ở bảng sau.

======================================== BẢNG ĐƠN HÌNH 1 ========================================
  Hệ số Cơ sở  Phương án      y1          y2         y3         y4      